# 16. HMM을 online으로 적용한다는 것의 의미와 방법

8번부터 12번까지 계속 "16번에서 다루겠다"고 미뤄둔 문제를 여기서 한꺼번에 정리할게. 순서는 이래.

1. online의 정확한 정의 (정보집합)
2. HMM 파이프라인에서 미래 정보가 새는 여섯 군데
3. 스무딩이 미래를 쓴다는 것의 수식적 증명
4. online 절차: 재추정 일정, 필터 이어가기
5. 라벨 정렬
6. 매매 시점 문제
7. JM의 online 적용
8. online과 offline의 차이 측정
9. 함정, 코드, 체크리스트

---

## 0단계: online의 정의 — "그 시점에 알 수 있었던 것만 쓴다"

t일까지 관측된 모든 데이터의 집합을 **정보집합** $\mathcal{F}_t$라고 하자.

$$
\mathcal{F}_t = \{r_1, r_2, \dots, r_t\}
$$

**online 분석의 정의:** t일에 내리는 모든 판단(레짐 확률, 파라미터, 표준화 기준, 매매 결정)이 **$\mathcal{F}_t$만으로 계산 가능해야** 한다. 수학적으로는 "$\mathcal{F}_t$-가측(measurable)"이라고 표현해.

**offline 분석:** 전체 표본 $\mathcal{F}_T$($T$는 마지막 날)를 한 번에 써서 과거의 모든 날을 판단해. 과거를 **이해**하는 데는 좋지만, 그 결과로 백테스트를 하면 "그때는 몰랐던 미래"를 쓴 셈이야.

비유하자면 offline은 **경기가 끝난 뒤 녹화 영상을 돌려보며 해설하는 것**이고, online은 **생중계로 해설하는 것**이야. 해설자가 "이 패스가 결승골로 이어집니다!"라고 말할 수 있는 건 녹화 영상에서만 가능하지.

---

## 1단계: HMM에서 미래 정보가 새는 여섯 군데

| # | 누수 지점 | offline (틀림) | online (맞음) |
|---|---|---|---|
| ① | 레짐 확률 | 스무딩 $P(S_t \mid \mathcal{F}_T)$ | 필터링 $P(S_t \mid \mathcal{F}_t)$ |
| ② | 모형 파라미터 | 전체 표본 추정 $\hat\theta_T$ | 그 시점까지 추정 $\hat\theta_t$ |
| ③ | 전처리 | 전체 표본 평균·표준편차로 표준화 | 그 시점까지의 통계로 표준화 |
| ④ | 레짐 라벨 | 한 번 추정한 라벨 | 재추정마다 뒤바뀔 수 있어 정렬 필요 |
| ⑤ | 매매 시점 | t일 종가 신호로 t일 수익률 사용 | t일 신호로 t+1일 수익률 사용 |
| ⑥ | 모형 설계 | 전체 결과를 보고 K, 피처를 고름 | 사전에 확정 |

8번에서 경고한 건 ①이었고, `statsmodels`의 필터링 확률을 쓰면 ①은 해결돼. **하지만 ②~⑥이 남아 있어.** 많은 백테스트가 ①만 고치고 나머지를 놓쳐. 하나씩 보자.

---

## 2단계: 스무딩이 미래를 쓴다는 수식적 증명 (①)

8번에서 "스무딩은 끝에서부터 거꾸로 계산한다"고만 했지. 그 계산식(Kim smoother)을 보면 미래를 쓴다는 게 명확해져.

$$
\gamma_t(i) = \xi_t(i)\sum_j \frac{p_{ij}\,\gamma_{t+1}(j)}{P(S_{t+1} = j \mid \mathcal{F}_t)}
$$

- $\xi_t(i)$: t일의 필터링 확률 (과거만 사용)
- $\gamma_{t+1}(j)$: **t+1일의 스무딩 확률** ← 미래 정보
- 분모: t일에 예측한 t+1일 레짐 확률 (8번 필터의 ① 예측 단계 값)

$\gamma_t$를 구하려면 $\gamma_{t+1}$이 필요하고, $\gamma_{t+1}$을 구하려면 $\gamma_{t+2}$가 필요하고… 결국 **마지막 날 T까지의 모든 데이터**가 들어가.

**숫자로 확인 (8번 예시 이어서).** 8번에서 −3% 당일 필터링은 $\xi_t = (0.03,\ 0.97)$, 다음 날 예측은 $(0.080,\ 0.920)$이었어. 이후 평온한 날이 계속 이어져서 **t+1일의 스무딩 확률이 (0.9, 0.1)이 됐다고 가정**해볼게(설명용 가정이야).

$$
\gamma_t(\text{평온}) \propto 0.03 \times \left(\frac{0.98 \times 0.9}{0.080} + \frac{0.02 \times 0.1}{0.920}\right) \approx 0.03 \times 11.03 = 0.331
$$

$$
\gamma_t(\text{혼란}) \propto 0.97 \times \left(\frac{0.05 \times 0.9}{0.080} + \frac{0.95 \times 0.1}{0.920}\right) \approx 0.97 \times 0.666 = 0.646
$$

정규화하면 $\gamma_t \approx (0.34,\ 0.66)$이야.

| | t일 혼란 확률 |
|---|---|
| t일 당시 (필터링) | **97%** |
| 이후 평온했다는 걸 알고 난 뒤 (스무딩) | **66%** |

**같은 날에 대한 판단이 미래를 알고 나서 97%에서 66%로 수정됐어.** 스무딩 결과로 "그날 레짐 판단은 적당히 신중했다"고 백테스트하면, 실제로는 97%로 확신했던 당시의 과민 반응이 가려져.

---

## 3단계: 파라미터도 online이어야 한다 (②)

필터링 확률은 이렇게 계산돼.

$$
\xi_t = P(S_t \mid r_1, \dots, r_t;\ \theta)
$$

데이터는 t일까지만 쓰지만, **파라미터 θ(μ, σ, P)를 어떻게 얻었는지**가 문제야. 전체 표본으로 추정한 $\hat\theta_T$를 넣으면:

- 2019년의 필터링 확률 계산에 **2020년 코로나 폭락으로 학습된** "혼란 레짐의 변동성" 값이 들어가.
- 2019년 당시의 모형은 코로나 수준의 혼란을 본 적이 없으니, 실제로는 레짐 구분 기준이 달랐을 거야.

그래서 올바른 online 레짐 확률은:

$$
\boxed{\xi_t^{\text{online}} = P\big(S_t \mid r_1, \dots, r_t;\ \hat\theta_{\tau(t)}\big), \qquad \tau(t) = t \text{ 이전의 마지막 재추정 시점}}
$$

**데이터도 과거만, 파라미터도 과거 데이터로만 추정한 것**이어야 해.

---

## 4단계: online 절차 — 재추정과 필터 이어가기

매일 파라미터를 다시 추정하는 건 계산 비용이 크고 불필요해. 보통 **정해진 주기(월, 분기)로 재추정**하고, 그 사이에는 고정된 파라미터로 필터만 돌려.

```
최소 학습 기간 확보 (예: 처음 3~5년은 레짐 신호 없음, burn-in)

각 재추정 시점 τ (예: 매월 말) 마다:
    1. 전처리 기준 계산: [시작, τ] 구간의 평균·표준편차        ← ③
    2. 파라미터 추정: [시작, τ] 데이터로 HMM 학습 → θ̂_τ       ← ②
    3. 라벨 정렬: 변동성 작은 레짐을 항상 0번으로                ← ④
    4. 다음 재추정 전까지 매일 t에 대해:
         [시작, t] 데이터에 θ̂_τ로 필터를 돌려 마지막 값 ξ_t 저장  ← ①
    5. τ를 한 달 밀고 반복

저장된 ξ_t를 날짜순으로 이어붙인 것 = online 레짐 확률
```

**4번에서 필터를 매번 처음부터 다시 돌리는 이유:** 파라미터가 $\hat\theta_\tau$로 바뀌었으니, 이전 파라미터로 계산한 확률을 이어받으면 서로 다른 모형이 섞여. 새 파라미터로 창 시작부터 다시 돌리고 **마지막 날 값만** 쓰는 게 일관적이야. 필터는 한 번 돌리는 데 T × K² 연산이라 매우 싸니까 부담이 없어.

**확장 vs 롤링 윈도우:** 12번과 같은 트레이드오프야. 15번에서 말했듯 위기 레짐은 드물어서, 롤링 윈도우로 오래된 위기를 버리면 혼란 레짐을 학습할 데이터가 사라져. 레짐 모형에는 **확장 윈도우가 기본 선택**이야.

**재추정 주기:** 너무 잦으면 파라미터가 흔들리면서 레짐이 인위적으로 바뀌고, 너무 드물면 새로운 시장 상태를 늦게 반영해. 월 또는 분기가 흔한 선택이야. 결과가 주기 선택에 민감한지 확인해보는 게 좋아.

---

## 5단계: 라벨 정렬 (④)

EM은 레짐 번호를 마음대로 붙여. 1월 재추정에서는 "레짐 0 = 평온"이었는데, 2월 재추정에서는 "레짐 0 = 혼란"이 될 수 있어. 정렬하지 않고 이어붙이면 **2월 1일에 가짜 레짐 전환**이 생겨.

**방법 1 — 기준 통계로 정렬 (추천):** 매 재추정마다 변동성 $\sigma_k$가 작은 순서로 번호를 다시 매겨. 15번에서 말했듯 통계적 레짐은 사실상 변동성 레짐이라, 이 기준이 가장 안정적이야.

**방법 2 — 이전 추정과 매칭:** 새 파라미터와 직전 파라미터 사이의 거리가 최소가 되도록 레짐을 짝지어. 레짐이 3개 이상이거나 변동성만으로 구분이 애매할 때 써.

정렬 후에도 **재추정 시점에 레짐 확률이 불연속적으로 튀는지** 반드시 그래프로 확인해봐. 튄다면 파라미터가 불안정하다는 신호야.

---

## 6단계: 매매 시점 문제 (⑤)

가장 흔하면서도 간과되는 누수야.

t일 종가까지의 수익률 $r_t$로 필터링 확률 $\xi_t$를 계산했다고 하자. 그런데 $r_t$는 **t일 장이 끝나야** 확정돼. 그러니 $\xi_t$로 내린 결정은 **t일 종가에는 실행할 수 없고**, 빨라야 t+1일에 실행돼.

$$
\text{올바른 성과:}\quad \text{포지션}_t = g(\xi_t), \qquad \text{수익}_{t+1} = \text{포지션}_t \times r_{t+1}
$$

$\text{포지션}_t \times r_t$로 계산하면, −3% 폭락일에 "혼란 레짐 → 매도"를 **그 폭락이 일어나기 전에** 한 걸로 계산하게 돼. 백테스트 성과가 극적으로 좋아 보이지만 완전한 허구야.

**한국 시장에서의 실무적 세부사항:**

- 정규장 마감 전 15:20~15:30에 종가 단일가 매매가 있어. 15:20까지의 데이터로 신호를 만들면 종가 매매가 가능하지만, 그러면 신호에 **15:20 가격**을 써야 해(종가가 아니라).
- 10분봉으로 만든 RV나 RSJ를 레짐 피처로 쓸 때도 마찬가지야. **당일 마지막 봉까지 쓴 피처는 다음 날 결정에만** 쓸 수 있어.
- 10번 Shu et al.이 "trading delay"를 반영했다는 게 이 문제를 처리했다는 뜻이야.

**예측 연구에서의 대응:** 매매가 아니라 "레짐별로 예측력이 다른가"를 볼 때도 원칙은 같아. t+1일 수익률을 예측할 때 쓰는 레짐은 $\xi_t$ 또는 한 단계 앞 예측 확률 $\xi_t P$여야 하고, $\xi_{t+1}$을 쓰면 안 돼. $\xi_{t+1}$에는 예측 대상인 $r_{t+1}$이 이미 들어가 있으니까. 15번 6단계의 "y 기준 선택" 문제가 여기서 가장 직접적인 형태로 나타나.

---

## 7단계: JM의 online 적용

10번에서 JM은 동적계획법으로 **전체 경로**를 최적화한다고 했지. 전체 경로 최적화는 과거 레짐을 미래 데이터로 수정한다는 뜻이야. 스무딩과 같은 문제지.

**좋은 소식:** 10번의 동적계획법에서 **앞쪽으로 계산하는 부분만** 쓰면 online 판단이 바로 나와.

$$
V_t(k) = \frac{1}{2}\|x_t - \theta_k\|^2 + \min_j\big[V_{t-1}(j) + \lambda\,\mathbf{1}\{j \neq k\}\big]
$$

$V_t(k)$는 "t일까지의 데이터로, t일이 레짐 k로 끝나는 경로의 최소 비용"이야. **$\mathcal{F}_t$만 사용**하지. 따라서:

$$
\boxed{s_t^{\text{online}} = \arg\min_k V_t(k)}
$$

이게 "오늘까지 본 데이터 기준으로 가장 그럴듯한 오늘 레짐"이야. 거꾸로 추적(backtracking)은 **과거 레짐을 다시 판단하는 단계**라 online에서는 쓰지 않아.

**HMM과의 대응 관계:**

| | 과거만 사용 (online) | 전체 사용 (offline) |
|---|---|---|
| HMM | 필터링 $\xi_t$ | 스무딩 $\gamma_t$, 비터비 경로 |
| JM | 전진 계산 $\arg\min_k V_t(k)$ | 역추적한 전체 경로 |

나머지(중심 θ와 λ의 주기적 재추정, 표준화, 라벨 정렬, 매매 시점)는 HMM과 똑같이 처리하면 돼.

---

## 8단계: online과 offline의 차이를 측정하라

online 결과만 보고하지 말고, **offline과 얼마나 다른지**도 보여주는 게 좋아. 그 차이 자체가 "이 레짐이 실시간으로 얼마나 쓸모 있는가"의 척도야.

**일치율:** 15번에서 본 것처럼

$$
\text{일치율} = \frac{1}{T}\sum_{t}\mathbf{1}\{\hat S_t^{\text{online}} = \hat S_t^{\text{offline}}\}
$$

**감지 지연:** offline이 "혼란 레짐 시작"으로 판단한 날로부터 online이 혼란으로 전환하기까지 며칠이 걸렸나. 에피소드별로 계산해서 분포를 봐.

**look-ahead 프리미엄:** 같은 전략을 offline 레짐과 online 레짐으로 각각 돌렸을 때의 성과 차이야.

$$
\text{look-ahead 프리미엄} = \text{성과}^{\text{offline}} - \text{성과}^{\text{online}}
$$

이 값이 크다면, offline 백테스트 성과의 상당 부분이 **미래를 알았기 때문에** 나온 거라는 뜻이야. 레짐 연구에서는 이 차이가 놀라울 정도로 큰 경우가 흔해.

---

## 9단계: 함정

**초기 구간(burn-in)의 문제.** 확장 윈도우의 첫 몇 년은 데이터가 적어서 파라미터가 불안정해. 게다가 한국 데이터를 2010년부터 시작하면, **2020년 이전의 모형은 코로나급 위기를 본 적이 없어.** 첫 대형 위기가 올 때 모형이 이를 제대로 분류하지 못할 수 있어. 이건 버그가 아니라 **실시간 분석의 현실**이야. 오히려 이게 정직한 결과지. 최소 학습 기간을 정하고, 그 이전은 분석에서 제외해.

**재추정 시점의 인위적 전환.** 5단계 라벨 정렬을 해도, 파라미터가 크게 바뀌면 재추정 날에 레짐 확률이 튈 수 있어. 레짐 전환 날짜와 재추정 날짜가 겹치는 빈도를 확인해봐. 우연보다 훨씬 자주 겹친다면 파라미터 불안정이 레짐 신호를 오염시키고 있는 거야.

**라이브러리의 기본값을 믿지 마.**
- `hmmlearn`의 `predict_proba`는 **스무딩** 확률이야.
- `statsmodels`의 `filtered_marginal_probabilities`는 필터링이지만, **전체 표본으로 추정한 파라미터**를 써.
- 어느 쪽이든 그대로 쓰면 ①이나 ②가 새.

**⑥ 모형 설계 단계의 누수.** 전체 데이터로 여러 K와 피처를 시도해보고 가장 해석이 잘 되는 걸 고른 다음, 그 설정으로 online 분석을 하면 **설계 자체에 미래 정보가 들어간** 거야. 코드로는 막을 수 없는 누수라서, 15번에서 말한 대로 **분석 전에 설정을 확정하고 기록**하는 것만이 해결책이야.

---

## 계산 코드: online HMM 파이프라인

```python
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from scipy.stats import norm

def fit_hmm(X, K=2, n_init=10):
    best, best_ll = None, -np.inf
    for seed in range(n_init):
        try:
            m = GaussianHMM(n_components=K, covariance_type='diag',
                            n_iter=300, random_state=seed).fit(X)
            ll = m.score(X)
            if ll > best_ll:
                best, best_ll = m, ll
        except Exception:
            continue
    # ④ 라벨 정렬: 변동성 작은 순서
    sig = np.sqrt(best.covars_.reshape(K, -1)[:, 0])
    order = np.argsort(sig)
    mu = best.means_[order, 0]
    P = best.transmat_[np.ix_(order, order)]
    return mu, sig[order], P

def hamilton_filter_last(r, mu, sig, P):
    """① 필터링: 주어진 데이터의 마지막 날 레짐 확률만 반환"""
    K = len(mu)
    evals, evecs = np.linalg.eig(P.T)                  # 정상분포로 초기화
    xi = np.real(evecs[:, np.argmin(np.abs(evals - 1))])
    xi = xi / xi.sum()
    for x in r:
        pred = xi @ P
        post = pred * norm.pdf(x, mu, sig)
        xi = post / post.sum()
    return xi

def online_hmm(ret: pd.Series, min_train=756, refit='M', K=2):
    ret = ret.dropna()
    dates = ret.index
    refit_dates = ret.resample(refit).last().index      # 매월 말 재추정
    out = pd.DataFrame(index=dates, columns=range(K), dtype=float)

    params = None
    for i in range(min_train, len(dates)):
        t = dates[i]
        hist = ret.iloc[:i + 1]                          # [시작, t] 만 사용

        if params is None or t in refit_dates:
            train = hist.values
            m, s = train.mean(), train.std()             # ③ t 시점까지의 통계
            mu, sig, P = fit_hmm(((train - m) / s).reshape(-1, 1), K)  # ②
            params = (m, s, mu, sig, P)

        m, s, mu, sig, P = params
        z = (hist.values - m) / s
        out.loc[t] = hamilton_filter_last(z, mu, sig, P)

    return out   # out.loc[t] = t일 장 마감 후 알 수 있는 레짐 확률

xi_online = online_hmm(kospi_ret)

# ⑤ 매매 시점: t일 확률로 t+1일 수익률 평가
signal = xi_online[1].shift(1)                          # 하루 지연
strategy_ret = (1 - signal.round()) * kospi_ret         # 혼란 레짐이면 현금 (예시)
```

(재추정 시점 사이에는 매일 필터를 처음부터 다시 돌리는데, 속도가 문제되면 직전 날의 ξ를 이어받아 하루치만 갱신해도 돼. 같은 파라미터 구간 안에서는 결과가 동일해.)

**JM online 버전 (10번 코드 확장):**

```python
def jm_online_states(X, theta, lam):
    """전진 계산만 사용: 각 t에서 argmin_k V_t(k)"""
    T, K = len(X), len(theta)
    loss = 0.5 * ((X[:, None, :] - theta[None, :, :]) ** 2).sum(-1)
    penalty = lam * (1 - np.eye(K))
    V = loss[0].copy()
    s_online = np.zeros(T, dtype=int)
    s_online[0] = V.argmin()
    for t in range(1, T):
        V = loss[t] + (V[:, None] + penalty).min(axis=0)
        s_online[t] = V.argmin()                        # 역추적 없음
    return s_online
```

10번 3단계 예시 데이터로 `assign_states`(offline)와 `jm_online_states`(online)를 비교해봐. 레짐 전환이 **몇 날 늦게** 잡히는지가 바로 online의 감지 지연이야.

---

## 네 프로젝트를 위한 online 체크리스트

분석 코드를 짤 때 아래를 하나씩 확인해.

- [ ] 레짐 확률은 **필터링**(HMM) 또는 **전진 계산**(JM)만 사용했는가
- [ ] 파라미터는 **그 시점까지의 데이터로** 주기적으로 재추정했는가
- [ ] 표준화, 분위 경계, 평균 제거 등 **모든 전처리**가 그 시점까지의 통계로 계산됐는가
- [ ] 재추정마다 **라벨 정렬**을 했고, 재추정일에 확률이 튀지 않는가
- [ ] t+1일 수익률 예측에 **t일까지의 레짐**만 썼는가 (하루 지연)
- [ ] 10분봉 피처가 **당일 마지막 봉까지** 쓰였다면 다음 날부터만 사용했는가
- [ ] **최소 학습 기간** 이전은 분석에서 제외했는가
- [ ] K, 피처, 재추정 주기를 **분석 전에 확정**하고 기록했는가
- [ ] online과 offline의 **일치율, 감지 지연, 성과 차이**를 함께 보고했는가

---

**한 줄 요약:** online 적용이란 **t일의 모든 판단(레짐 확률, 파라미터, 전처리, 매매)을 t일까지 알 수 있었던 정보만으로 하는 것**이야. 스무딩 대신 필터링을 쓰는 건 시작일 뿐이고, 파라미터 주기적 재추정, 전처리 기준, 라벨 정렬, 하루 매매 지연까지 모두 지켜야 진짜 online이야. JM은 동적계획법의 전진 계산 결과만 쓰면 online이 돼. online과 offline의 성과 차이(look-ahead 프리미엄)를 함께 보고하면 결과의 신뢰도가 크게 올라가.

다음 17번 **Rank IC**는 레짐에서 벗어나, 4번 semibeta나 3번 RSJ 같은 **횡단면 시그널의 예측력을 평가하는 지표**야. 준비되면 말해줘.

Question : 계속 필터링, 필터링 확률 이런 말을 하는데 그게 뭐야? 잘 이해가 안 돼 그 개념에 대해서

---

# 필터링(Filtering)이란?

지금까지 이 단어를 설명 없이 너무 많이 썼어. 수식부터 들어가지 않고, 일상적인 예시로 개념을 먼저 잡은 다음 시장으로 옮겨올게.

---

## 1단계: 문제 상황 — 보이지 않는 것을 추측하기

네가 **창문이 없는 사무실**에서 일한다고 해보자. 밖에 비가 오는지 직접 볼 수 없어. 알 수 있는 단서는 하나뿐이야. **매일 아침 출근하는 동료가 우산을 들고 왔는가.**

- 숨겨진 상태: 오늘 비가 오는가 (직접 못 봄)
- 관측: 동료의 우산 (매일 봄)

그런데 이 단서는 **불완전해.** 비가 와도 동료가 우산을 깜빡할 수 있고, 비가 안 와도 혹시 몰라 들고 올 수 있어. 그래서 우산만 보고 "비가 온다"고 **확신할 수는 없고, 확률로만 추측**할 수 있어.

이게 정확히 시장 레짐 문제와 같은 구조야.

| 사무실 | 시장 |
|---|---|
| 비가 오는가 (숨겨짐) | 평온장인가 혼란장인가 (숨겨짐) |
| 동료의 우산 (관측) | 오늘의 수익률 (관측) |
| 우산은 불완전한 단서 | 수익률도 불완전한 단서 (평온장에도 가끔 −2%가 나옴) |

---

## 2단계: 숨겨진 상태에 대해 던질 수 있는 세 가지 질문

오늘이 3일째라고 하자. 네가 던질 수 있는 질문은 세 종류야.

```
         1일    2일    3일(오늘)    4일(내일)
관측:     ☂      ☂       ✗           ?
```

| 질문 | 쓰는 정보 | 이름 |
|---|---|---|
| **내일** 비가 올까? | 오늘까지의 우산 | **예측 (prediction)** |
| **오늘** 비가 오고 있을까? | 오늘까지의 우산 | **필터링 (filtering)** |
| **어제(2일)** 비가 왔을까? | 오늘까지의 우산 (어제 이후 정보 포함) | **스무딩 (smoothing)** |

수식으로 쓰면 이래. $\mathcal{F}_t$는 t일까지 관측한 모든 정보야.

$$
\text{예측: } P(S_{t+1} \mid \mathcal{F}_t) \qquad \text{필터링: } P(S_t \mid \mathcal{F}_t) \qquad \text{스무딩: } P(S_s \mid \mathcal{F}_t),\ s < t
$$

**필터링 확률(filtered probability)**은 필터링으로 얻은 결과값이야. 즉 **"오늘까지 본 정보로 판단한, 오늘 상태에 대한 확률"**이지.

세 개의 핵심 차이는 **"판단하려는 날"과 "정보가 끝나는 날"의 관계**야.

- 예측: 판단하는 날이 정보보다 **뒤** (미래를 추측)
- 필터링: 판단하는 날과 정보가 끝나는 날이 **같음** (지금을 추측)
- 스무딩: 판단하는 날이 정보보다 **앞** (과거를 되돌아봄. 그 이후 정보까지 활용)

---

## 3단계: 왜 "필터"라고 부르나

원래 **신호처리** 분야에서 온 용어야. 라디오 신호에 잡음이 섞여 있을 때, 잡음을 걸러내고(filter) 진짜 신호를 추정하는 걸 필터링이라고 불렀어.

커피 필터를 생각하면 돼. 관측값(우산, 수익률)은 **진짜 상태 + 잡음**이 섞인 흙탕물이고, 필터는 거기서 잡음을 걸러내 **진짜 상태에 대한 추정**만 뽑아내는 장치야.

그리고 결정적인 특징이 하나 있어. 라디오는 **실시간으로** 소리를 내야 하니까, 신호처리 필터는 **지금까지 들어온 신호만** 써야 해. 미래의 신호를 기다릴 수 없지. 그래서 "필터링"이라는 말에는 자연스럽게 **"과거와 현재만 쓴다"**는 의미가 담겨 있어.

---

## 4단계: 필터링이 계산되는 방식 — 매일 두 단계 반복

설정을 숫자로 정하자.

**날씨의 지속성 (전이확률):**
- 오늘 비 → 내일도 비: 70%
- 오늘 맑음 → 내일도 맑음: 70%

**우산이라는 단서의 정확도 (가능도):**
- 비 오는 날 우산을 들고 올 확률: 90%
- 맑은 날 우산을 들고 올 확률: 20%

첫날 아침엔 아무 정보가 없으니 비/맑음을 50:50으로 시작해.

필터링은 매일 두 단계를 반복해.

- **① 예측:** 어제의 판단을 날씨의 지속성으로 하루 굴린다. "어제 이 정도로 비가 왔다고 봤으니, 오늘은 아마 이 정도겠지."
- **② 갱신:** 오늘의 단서(우산)를 보고 판단을 고친다. "근데 우산을 들고 왔네? 그럼 비 쪽으로 좀 더 기울자."

### 1일: 우산 ☂

**① 예측:** 시작이 50:50이니 그대로 **비 0.5, 맑음 0.5**

**② 갱신:** "그 날씨였다면 이 단서가 나올 확률"(가능도)을 곱해.
- 비: 0.5 × 0.9 = 0.45
- 맑음: 0.5 × 0.2 = 0.10

합이 0.55니까, 나눠서 합을 1로 맞추면:

$$
\text{비} = \frac{0.45}{0.55} = \mathbf{0.818}, \qquad \text{맑음} = 0.182
$$

**1일의 필터링 확률: 비 81.8%**

### 2일: 우산 ☂

**① 예측:** 어제 판단(0.818, 0.182)을 지속성으로 굴려.
- 오늘 비 = 어제 비였고 계속 비 + 어제 맑았는데 비로 바뀜 = $0.818 \times 0.7 + 0.182 \times 0.3 = 0.627$
- 오늘 맑음 = $0.373$

**② 갱신:**
- 비: 0.627 × 0.9 = 0.564
- 맑음: 0.373 × 0.2 = 0.075
- 정규화: 비 = 0.564 / 0.639 = **0.883**

**2일의 필터링 확률: 비 88.3%**

### 3일: 우산 없음 ✗

**① 예측:**
- 비 = $0.883 \times 0.7 + 0.117 \times 0.3 = 0.653$
- 맑음 = $0.347$

**② 갱신:** 이번엔 "우산이 **없을** 확률"을 곱해야 해. 비 오는 날 없을 확률은 10%, 맑은 날 없을 확률은 80%야.
- 비: 0.653 × 0.1 = 0.065
- 맑음: 0.347 × 0.8 = 0.278
- 정규화: 비 = 0.065 / 0.343 = **0.190**

**3일의 필터링 확률: 비 19.0%**

### 정리

| | 1일 | 2일 | 3일 |
|---|---|---|---|
| 관측 | ☂ | ☂ | ✗ |
| ① 예측 후 비 확률 | 50% | 62.7% | 65.3% |
| ② 갱신 후 비 확률 (**필터링 확률**) | **81.8%** | **88.3%** | **19.0%** |

두 가지를 관찰해봐.

- **① 예측 단계는 판단을 "관성" 쪽으로 끌어당겨.** 2일에 아직 우산을 보기 전인데도 비 확률이 62.7%로 시작해. 어제 비가 왔으면 오늘도 올 가능성이 크니까.
- **② 갱신 단계는 오늘의 증거로 판단을 움직여.** 3일에 예측은 비 65.3%였지만, 우산이 없는 걸 보고 19.0%로 확 떨어졌어.

필터링 확률은 이 **"관성"과 "새 증거"의 줄다리기** 결과야.

---

## 5단계: 수식으로 정리

방금 계산한 걸 일반적인 수식으로 쓰면 이렇게 돼. $\xi_t(j)$는 t일의 필터링 확률(상태 j일 확률)이야.

**① 예측:**

$$
P(S_t = j \mid \mathcal{F}_{t-1}) = \sum_i \xi_{t-1}(i)\cdot p_{ij}
$$

"어제 상태 i였을 확률 × i에서 j로 바뀔 확률"을 모든 i에 대해 더한 거야. 2일의 $0.818 \times 0.7 + 0.182 \times 0.3$이 정확히 이 계산이었어.

**② 갱신 (베이즈 정리):**

$$
\xi_t(j) = \frac{P(S_t = j \mid \mathcal{F}_{t-1})\times P(\text{오늘 관측} \mid S_t = j)}{\sum_k P(S_t = k \mid \mathcal{F}_{t-1})\times P(\text{오늘 관측} \mid S_t = k)}
$$

- 분자: **사전 믿음(예측) × 가능도**
- 분모: 모든 상태에 대해 분자를 더한 값. 확률의 합을 1로 맞추는 역할이야.

베이즈 정리를 한 줄로 말하면 "**새 믿음 ∝ 이전 믿음 × 증거가 그 믿음과 맞는 정도**"야.

---

## 6단계: 스무딩은 어떻게 다른가 — 같은 예시로

이제 3일 저녁이 됐어. **2일에 비가 왔는지** 다시 생각해보자.

- 2일 당시의 판단(필터링): **88.3%**
- 그런데 이제 3일에 우산이 없었다는 걸 알아. 3일에 맑았다면, 날씨의 지속성 때문에 **2일도 맑았을 가능성이 조금 더 높아지지.**

계산해보면(3일 관측이 각 경우에 얼마나 잘 맞는지를 2일 판단에 곱해주는 방식):

- 2일이 비였다면 3일에 우산이 없을 확률: $0.7 \times 0.1 + 0.3 \times 0.8 = 0.31$
- 2일이 맑았다면 3일에 우산이 없을 확률: $0.3 \times 0.1 + 0.7 \times 0.8 = 0.59$

2일의 필터링 확률에 이걸 곱해서 정규화하면:

- 비: $0.883 \times 0.31 = 0.274$
- 맑음: $0.117 \times 0.59 = 0.069$
- 비 = $0.274 / 0.343 = $ **0.799**

| 2일에 비가 왔을 확률 | 값 |
|---|---|
| 2일 당시 판단 (필터링) | **88.3%** |
| 3일 정보까지 보고 다시 판단 (스무딩) | **79.9%** |

**같은 2일에 대한 판단인데, 그 이후 정보를 보고 나서 바뀌었어.** 이게 스무딩이야. 과거를 더 정확하게 이해하는 데는 좋지만, **2일 아침에 이 79.9%라는 숫자를 알 수는 없었어.** 3일의 정보가 필요했으니까.

그래서 "2일에 비가 올 것 같으면 택시를 탄다" 같은 결정의 성과를 평가할 때, 79.9%(스무딩)를 쓰면 **미래를 본 거야.** 88.3%(필터링)를 써야 그때 실제로 내릴 수 있었던 결정이 돼. 16번에서 계속 강조한 게 바로 이거야.

---

## 7단계: 시장으로 옮기기

8번의 HMM 숫자 예시를 다시 보면 **완전히 같은 계산**이야.

| 우산 예시 | 시장 레짐 예시 (8번) |
|---|---|
| 비 / 맑음 | 혼란장 / 평온장 |
| 날씨 지속성 70% | 레짐 유지 확률 98%, 95% |
| 우산 가능도 (90%, 20%) | 정규분포 밀도 $f_k(r_t)$ |
| 오늘 우산 없음 | 오늘 수익률 −3% |
| 필터링 확률 | "오늘 장 마감 후 알 수 있는 혼란장 확률" |

유일한 차이는 가능도야. 우산은 있다/없다 두 가지라 확률을 표로 줬지만, 수익률은 연속적인 숫자라서 **정규분포 밀도함수**로 "이 레짐이었다면 이 수익률이 얼마나 자연스러운가"를 계산해. 8번에서 −3%가 평온장(σ = 0.8%)에서는 거의 불가능하고, 혼란장(σ = 2.5%)에서는 흔한 값이었지. 그게 우산 예시의 "비 오는 날 우산 90%, 맑은 날 20%"와 같은 역할이야.

**그래서 필터링 확률을 한 문장으로 말하면:**

> **"오늘 장이 끝난 시점에, 오늘까지의 수익률만 보고 판단한 '지금 혼란장일 확률'"**

뉴스 앵커가 오늘 저녁 방송에서 말할 수 있는 판단이야. 한 달 뒤에 돌아보며 하는 해설(스무딩)이 아니라.

---

## 8단계: 참고 — 이름만 알아두면 좋은 것

**Hamilton filter:** 8, 9번에서 이 이름이 나왔지. 레짐처럼 **몇 개의 이산 상태**에 대해 지금 본 필터링을 하는 것을 금융·계량경제학에서 이렇게 불러. Markov switching 모형을 도입한 Hamilton의 이름을 딴 거야. 계산 내용은 4단계와 똑같아.

**Kalman filter:** 숨겨진 상태가 이산(비/맑음)이 아니라 **연속적인 숫자**일 때(예: 숨겨진 "진짜 변동성"이 매일 조금씩 움직이는 경우) 쓰는 필터야. 원리는 똑같이 "예측 → 갱신"의 반복이야. 나중에 어디선가 보게 되면 "아, 연속 버전의 필터링이구나" 하고 이해하면 돼.

---

**한 줄 요약:** 필터링은 **"오늘까지의 관측만으로 오늘의 숨겨진 상태를 확률로 추정하는 것"**이고, 필터링 확률은 그 결과값이야. 매일 ① 어제 판단을 지속성으로 굴리고(예측), ② 오늘 관측으로 베이즈 정리를 써서 고치는(갱신) 과정을 반복해서 계산해. 스무딩은 그 이후 정보까지 써서 과거를 다시 판단하는 거라, 과거 이해에는 좋지만 백테스트에 쓰면 미래를 본 게 돼.

이제 16번 내용이 다시 읽힐 거야. 막히는 부분이 있으면 말해주고, 괜찮으면 17번 Rank IC로 넘어가자.